# 1. Read source data

In [65]:
import sys
import numpy as np
import pandas as pd
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)
from config import RESULT_COLUMNS


In [66]:
from mhcgnomes import parse
DATA_PATH = "D:/Google Drive/Work/ITMO/data/schreiber_full/"
PREDICTION_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/schreiber_full/"
# read total data

source_total_df =  pd.read_csv("{}schreiber_total_data_full.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(source_total_df['allele']))
per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt = pd.read_csv("{}schreiber_data_full_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    df_wt = pd.read_csv("{}schreiber_data_full_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";") 
    
    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt
    
    assert 'peptide' in df_mt.columns
    assert 'length' in df_mt.columns
    assert RESULT_COLUMNS['id'] in df_mt.columns
    assert 'peptide' in df_wt.columns
    assert 'length' in df_wt.columns
    assert RESULT_COLUMNS['id'] in df_wt.columns
    
    
def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None


def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None
ALLELES

['H-2-Dd', 'H-2-IEd', 'H-2-Ld', 'H-2-IAd', 'H-2-Kd']

# 3. Read Joined predictions

In [67]:
per_allele_predictions = dict()


additional_info_columns = ['VAR_ID', 'mutant_id', 'transcript_id', 'peptide_offset',
                        'A20_RGT', 'A20_RDP', 'A20_RAD_REF',
                        'A20_RAD_ALT', 'A20_RAF_ALT', 'A20_RPKM_mut',  'source_sequence_name', 
                        'peptide_wt', 'peptide_mt' , 'allele',  'length'
                       ]
per_allele_all_methods = {allele_name : set() for allele_name in ALLELES}
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df = pd.read_csv("{}total_predictions_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";")
    per_allele_predictions[allele_name] = total_df
    per_allele_all_methods[allele_name].update(set([item.split('_')[0] for item in total_df.columns if  item not in additional_info_columns]))

    
    
class_I_total_df = pd.read_csv("{}total_predictions_class_I.csv".format(PREDICTION_PATH),sep=";" )
class_II_total_df  = pd.read_csv("{}total_predictions_class_II.csv".format(PREDICTION_PATH),sep=";")

In [68]:
total_df.columns

Index(['mutant_id', 'source_sequence_name', 'allele', 'length', 'peptide_mt',
       'netMHC-4.0_percentile_rank_mt', 'netMHCpan-4.1_percentile_rank_ba_mt',
       'netMHCpan-4.1_percentile_rank_el_mt', 'ghmm_percentile_rank_mt',
       'mhcflurry_percentile_rank_ba_mt', 'mhcflurry_percentile_rank_el_mt',
       'mhcnuggets_percentile_rank_mt', 'puffin_percentile_rank_mt',
       'peptide_wt', 'netMHC-4.0_percentile_rank_wt',
       'netMHCpan-4.1_percentile_rank_ba_wt',
       'netMHCpan-4.1_percentile_rank_el_wt', 'ghmm_percentile_rank_wt',
       'mhcflurry_percentile_rank_ba_wt', 'mhcflurry_percentile_rank_el_wt',
       'mhcnuggets_percentile_rank_wt', 'puffin_percentile_rank_wt', 'VAR_ID',
       'transcript_id', 'peptide_offset', 'A20_RGT', 'A20_RDP', 'A20_RAD_REF',
       'A20_RAD_ALT', 'A20_RAF_ALT', 'A20_RPKM_mut', 'netMHC-3.4_score_mt',
       'netMHC-3.4_affinity_mt', 'netMHC-4.0_score_mt',
       'netMHC-4.0_affinity_mt', 'netMHCpan-4.1_score_ba_mt',
       'netMHCpan-4.1_

In [69]:
per_allele_all_methods

{'H-2-Dd': {'ghmm',
  'mhcflurry',
  'mhcnuggets',
  'netMHC-3.4',
  'netMHC-4.0',
  'netMHCpan-4.1',
  'puffin'},
 'H-2-IEd': {'ghmm', 'mhcnuggets', 'netMHCIIpan-4.0', 'puffin'},
 'H-2-Ld': {'ghmm',
  'mhcflurry',
  'mhcnuggets',
  'netMHC-3.4',
  'netMHC-4.0',
  'netMHCpan-4.1',
  'puffin'},
 'H-2-IAd': {'ghmm', 'mhcnuggets', 'netMHCIIpan-4.0', 'puffin'},
 'H-2-Kd': {'ghmm',
  'mhcflurry',
  'mhcnuggets',
  'netMHC-3.4',
  'netMHC-4.0',
  'netMHCpan-4.1',
  'puffin'}}

# 4. Apply filtering

In [70]:
def filter_epitopes(df, method_name, advantage = False):
    if method_name == 'netMHC-3.4':
        df = df.loc[df[method_name + "_affinity_mt"] < 500].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    elif  method_name == 'netMHC-4.0':
        df = df.loc[df[method_name +  "_percentile_rank_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_advantage"] > 1].copy()
    elif method_name == 'netMHCpan-4.1':
        df = df.loc[df[method_name + "_percentile_rank_el_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_el_advantage"] > 1].copy()
    elif method_name == 'netMHCIIpan-4.0':
        df = df.loc[df[method_name + "_percentile_rank_el_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_el_advantage"] > 1].copy()
    elif method_name == 'mhcflurry':
        df = df.loc[df[method_name + "_percentile_rank_el_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_el_advantage"] > 1].copy()
    elif method_name == 'mhcnuggets':
        df = df.loc[df[method_name + "_percentile_rank_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_advantage"] > 1].copy()
    elif method_name == 'acme':
        df = df.loc[df[method_name + "_score_mt"] > 0.42].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    elif method_name == 'mhcseqnet':
        df = df.loc[df[method_name + "_score_mt"] > 0.99].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    elif method_name == 'puffin':
        df = df.loc[df[method_name + "_percentile_rank_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_advantage"] > 1].copy()
#     elif method_name == 'puffin':
#         df = df.loc[df[method_name + "_binding_likelihood_mt"] > 0.5].copy()
#         if advantage:
#             df = df.loc[df[method_name + "_binding_likelihood_advantage"] > 1].copy()
    elif method_name == 'mhcattnnet':
        df = df.loc[df[method_name + "_score_mt"] >= 1].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    return df


In [71]:
def per_allele_filter_predictions(per_allele_df, selected_methods, advantage=False):
    per_allele_result = dict()
    for allele_name in ALLELES:
        prediction_df =  per_allele_df[allele_name]
        count = len(prediction_df)
        print(f"{allele_name}:({count}) ->", end=" ")
        for method in selected_methods:
            if method in per_allele_all_methods[allele_name]:
                prediction_df = filter_epitopes(prediction_df, method, advantage)
                count = len(prediction_df)
                print(f"{method}:({count}) ->", end=" ")
        print()
        per_allele_result[allele_name] = prediction_df
    return per_allele_result

    
# def reorder_result_columns(df_filtered):  
#     df_filtered = df_filtered.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
#      df_filtered = df_filtered.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])

#     return df_filtered


def filter_and_save_to_file(per_region_per_allele_predictions, selected_methods, advantage=False):
    print("1.Filter using methods")
    per_allele_top_epitopes = dict()
    
    per_allele_top_epitopes = per_allele_filter_predictions(per_allele_predictions, selected_methods, advantage=advantage)
    
    print("2.Write to files")
    #DATA_PATH = "/home/denklewer/Data/gbm122/" #ubuntu machine
    DATA_PATH = PREDICTION_PATH
    for allele_name, df_filtered in per_allele_top_epitopes.items():
        if advantage:
            df_filtered.to_csv("{}top_epitopes_{}_filtered+advantage_{}.csv".format(DATA_PATH, allele_name, "_".join(selected_methods)),sep=";", index=False)
        else:
            df_filtered.to_csv("{}top_epitopes_filtered_all_regions_{}.csv".format(DATA_PATH,allele_name,  "_".join(selected_methods)),sep=";", index=False)


In [72]:
selected_methods = ['netMHCIIpan-4.0', 'netMHCpan-4.1', 'mhcflurry','mhcnuggets','puffin']
dicc  = filter_and_save_to_file(per_allele_predictions, selected_methods, advantage=False)

1.Filter using methods
H-2-Dd:(8738) -> netMHCpan-4.1:(187) -> mhcflurry:(21) -> mhcnuggets:(3) -> puffin:(3) -> 
H-2-IEd:(52091) -> netMHCIIpan-4.0:(491) -> mhcnuggets:(77) -> puffin:(23) -> 
H-2-Ld:(8738) -> netMHCpan-4.1:(197) -> mhcflurry:(48) -> mhcnuggets:(15) -> puffin:(14) -> 
H-2-IAd:(52091) -> netMHCIIpan-4.0:(344) -> mhcnuggets:(57) -> puffin:(44) -> 
H-2-Kd:(8738) -> netMHCpan-4.1:(156) -> mhcflurry:(42) -> mhcnuggets:(19) -> puffin:(15) -> 
2.Write to files


In [73]:
dicc  = filter_and_save_to_file(per_allele_predictions, selected_methods, advantage=True)

1.Filter using methods
H-2-Dd:(8738) -> netMHCpan-4.1:(105) -> mhcflurry:(14) -> mhcnuggets:(2) -> puffin:(2) -> 
H-2-IEd:(52091) -> netMHCIIpan-4.0:(181) -> mhcnuggets:(11) -> puffin:(2) -> 
H-2-Ld:(8738) -> netMHCpan-4.1:(110) -> mhcflurry:(26) -> mhcnuggets:(4) -> puffin:(4) -> 
H-2-IAd:(52091) -> netMHCIIpan-4.0:(142) -> mhcnuggets:(25) -> puffin:(24) -> 
H-2-Kd:(8738) -> netMHCpan-4.1:(90) -> mhcflurry:(25) -> mhcnuggets:(9) -> puffin:(6) -> 
2.Write to files
